# Video -> Gaussian Splat (free Colab test)
Runs `pipeline/run.sh` from [realestatewalkthrough](https://github.com/tschnelli/realestatewalkthrough) on the sample video.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.
Cells 2 and 4 take a while (installs, training). If a cell fails, copy the error text back.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install COLMAP, ffmpeg and nerfstudio (several minutes)
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip install -q nerfstudio
!colmap -h | head -1 && ns-train --help | head -1

In [ ]:
# 3. Get the project (and sample video)
!rm -rf /content/rew && git clone -q https://github.com/tschnelli/realestatewalkthrough /content/rew
!ls -la /content/rew/samples

In [ ]:
# 4. Run the pipeline: frames -> COLMAP poses -> splatfacto training -> export
# Apt COLMAP has no CUDA, so feature matching runs on CPU (--no-gpu). Fine for a short low-res clip.
# 7000 iterations keeps the free T4 session short; raise ITERS for better quality.
import os
os.environ["QT_QPA_PLATFORM"] = "offscreen"      # COLMAP is headless here
os.environ["ITERS"] = "7000"
os.environ["PROCESS_ARGS"] = "--no-gpu --sfm-tool colmap"
!cd /content/rew && bash pipeline/run.sh samples/cabin-walkthrough.mov /content/out 150

In [ ]:
# 5. Check how many frames COLMAP registered (low numbers = poor capture)
import json, glob
f = glob.glob('/content/out/data/transforms.json')
t = json.load(open(f[0])) if f else {}
print('registered frames:', len(t.get('frames', [])))

In [ ]:
# 6. Download the splat, then drop it in viewer/ as scene.ply and open viewer/?src=scene.ply
from google.colab import files
import glob
ply = sorted(glob.glob('/content/out/export/*.ply'))
print(ply)
files.download(ply[0])